# Interactive EGFR Cascade Simulation

## Light Pattern Builder

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

class LightPatternBuilder:
    def __init__(self, t_max=100):
        self.style = {"description_width": "70px"}
        self.slider_layout = widgets.Layout(width="280px")
        self.t_max_slider = widgets.FloatSlider(
            value=t_max, min=10, max=500, step=5,
            description="t_max", style=self.style, layout=self.slider_layout,
        )
        self.pulses = []
        self.pulse_container = widgets.VBox()
        self.plot_out = widgets.Output()

        add_btn = widgets.Button(description="+ Add pulse", button_style="success",
                                  layout=widgets.Layout(width="120px"))
        add_btn.on_click(lambda _: self._add_pulse())

        save_btn = widgets.Button(description="Save pattern", button_style="primary",
                                   layout=widgets.Layout(width="120px"))
        save_btn.on_click(self._save)
        self.save_out = widgets.Output()

        top = widgets.HBox([self.t_max_slider, add_btn, save_btn, self.save_out])
        self.ui = widgets.VBox([top, self.pulse_container, self.plot_out])

        self.t_max_slider.observe(lambda _: self._refresh_plot(), names="value")
        self._add_pulse(t_on=0, t_off=5, amplitude=1.0)

    def _add_pulse(self, t_on=None, t_off=None, amplitude=1.0):
        idx = len(self.pulses)
        t_on = t_on if t_on is not None else idx * 15
        t_off = t_off if t_off is not None else t_on + 5

        on_s = widgets.FloatSlider(value=t_on, min=0, max=500, step=0.5,
                                    description="t_on", style=self.style, layout=self.slider_layout)
        off_s = widgets.FloatSlider(value=t_off, min=0, max=500, step=0.5,
                                     description="t_off", style=self.style, layout=self.slider_layout)
        amp_s = widgets.FloatSlider(value=amplitude, min=0, max=5, step=0.05,
                                     description="amplitude", style=self.style, layout=self.slider_layout)
        rm_btn = widgets.Button(description="x", button_style="danger",
                                 layout=widgets.Layout(width="35px"))

        pulse = {"on": on_s, "off": off_s, "amp": amp_s, "btn": rm_btn}
        self.pulses.append(pulse)

        for s in (on_s, off_s, amp_s):
            s.observe(lambda _: self._refresh_plot(), names="value")

        rm_btn.on_click(lambda _, p=pulse: self._remove_pulse(p))
        row = widgets.HBox([widgets.Label(f"#{idx+1}", layout=widgets.Layout(width="30px")),
                            on_s, off_s, amp_s, rm_btn])
        pulse["row"] = row
        self._rebuild_container()
        self._refresh_plot()

    def _remove_pulse(self, pulse):
        if pulse in self.pulses:
            self.pulses.remove(pulse)
            self._rebuild_container()
            self._refresh_plot()

    def _rebuild_container(self):
        for i, p in enumerate(self.pulses):
            p["row"].children[0].value = f"#{i+1}"
        self.pulse_container.children = [p["row"] for p in self.pulses]

    def get_pulses(self):
        return [{"t_on": p["on"].value, "t_off": p["off"].value, "amplitude": p["amp"].value}
                for p in self.pulses]

    def light_func(self, t_val, t_args=None):
        """Evaluate the light pattern at time t_val. Compatible with Model API."""
        pulses = t_args.get("pulses") if t_args and "pulses" in t_args else self.get_pulses()
        return sum(p["amplitude"] for p in pulses if p["t_on"] <= t_val <= p["t_off"])

    def _refresh_plot(self):
        pulses = self.get_pulses()
        t_max = self.t_max_slider.value
        times = np.linspace(0, t_max, 1000)
        vals = [self.light_func(tv, {"pulses": pulses}) for tv in times]

        with self.plot_out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 1.8))
            ax.fill_between(times, vals, alpha=0.35, color="gold")
            ax.plot(times, vals, color="orange", linewidth=1)
            ax.set_ylabel("Light")
            ax.set_xlabel("Time")
            ax.set_xlim(0, t_max)
            ax.set_ylim(bottom=-0.05)
            plt.tight_layout()
            plt.show()

    def _save(self, _):
        pattern = {"t_max": self.t_max_slider.value, "pulses": self.get_pulses()}
        path = "light_pattern.json"
        with open(path, "w") as f:
            json.dump(pattern, f, indent=2)
        with self.save_out:
            clear_output(wait=True)
            print(f"Saved to {path}")

    def display(self):
        display(self.ui)

light_builder = LightPatternBuilder()
light_builder.display()

In [7]:
import pandas as pd
df = pd.read_parquet('./dataset.parquet')
df['ramp_pattern_name'].unique()

array(['3-2-1minIntervals', 'Single', 'Sustained', 'ramp1'], dtype=object)

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

from model.mechanistic.mechanistic_model import Model
from model.mechanistic.egfr_simplified import model_eqs, PARAM_NAMES, STATE_NAMES, NODE_NAMES

In [ ]:
def light_pulse(t_val, t_args=None):
    """Light on from t_on to t_off, off otherwise."""
    t_on = t_args.get("t_on", 0) if t_args else 0
    t_off = t_args.get("t_off", 5) if t_args else 5
    return 1.0 if t_on <= t_val <= t_off else 0.0

m = Model(
    name="egfr_interactive",
    states=STATE_NAMES,
    parameters=PARAM_NAMES,
    model_definition=model_eqs,
    t_dep="light",
    t_func=light_pulse,
)
system = m.make_numerical()

to_hand_pick = 'egfr_params.json'

In [ ]:
DEFAULTS = {
    "Km": 0.1,
    # RAS: fast activation by light, strong GAP-mediated deactivation
    "k12": 0.8,  "k21": 1.5,
    # RAF: moderate activation by RAS, NFB-sensitive deactivation
    "k34": 0.5,  "knfb": 0.8, "k43": 0.3,
    # MEK: dual-phosphorylation by RAF, PP2A deactivation
    "k56": 0.6,  "k65": 1.2,
    # ERK: dual-phosphorylation by MEK, phosphatase deactivation
    "k78": 0.5,  "k87": 1.0,
    # NFB (DUSP/Sprouty-like): slow transcriptional feedback from ERK
    "f12": 0.08, "f21": 0.05,
}
if to_hand_pick is not None: 
    with open(to_hand_pick, 'r') as f:
        DEFAULTS = json.load(f)
print(f"Using {to_hand_pick or 'Default'} parameters: {DEFAULTS}")

COLORS = {"RAS": "#e41a1c", "RAF": "#377eb8", "MEK": "#4daf4a", "NFB": "#984ea3", "ERK": "#ff7f00"}

def simulate_and_plot(**kwargs):
    t_end = kwargs.pop("t_end")
    t_on = kwargs.pop("t_on")
    t_off = kwargs.pop("t_off")

    params_vec = np.array([kwargs[p] for p in PARAM_NAMES])
    times = np.linspace(0, t_end, 500)
    y0 = np.zeros(len(STATE_NAMES))
    t_args = {"t_on": t_on, "t_off": t_off}

    sol = solve_ivp(
        lambda t, y: system(t, y, params_vec, light_pulse, t_args),
        [0, t_end], y0, t_eval=times, method="LSODA", rtol=1e-8,
    )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), height_ratios=[1, 4],
                                    sharex=True, gridspec_kw={"hspace": 0.05})

    # Light stimulus
    light_vals = [light_pulse(tv, t_args) for tv in times]
    ax1.fill_between(times, light_vals, alpha=0.3, color="gold")
    ax1.set_ylabel("Light")
    ax1.set_ylim(-0.05, 1.15)
    ax1.set_yticks([0, 1])

    # State trajectories
    if sol.success:
        for i, node in enumerate(NODE_NAMES):
            y_plot = sol.y[i]
            if node == "NFB":
                nfb_max = np.max(y_plot)
                if nfb_max > 1e-12:
                    y_plot = y_plot / nfb_max
                label = "NFB (norm)"
            else:
                label = node
            ax2.plot(times, y_plot, label=label, color=COLORS[node], linewidth=2)
        ax2.legend(loc="upper right")
    else:
        ax2.text(0.5, 0.5, f"Solver failed: {sol.message}", transform=ax2.transAxes,
                 ha="center", color="red", fontsize=12)

    ax2.set_xlabel("Time")
    ax2.set_ylabel("Active fraction")
    ax2.set_ylim(bottom=-0.02)
    plt.tight_layout()
    plt.show()

In [ ]:
style = {"description_width": "60px"}

param_sliders = {
    p: widgets.FloatLogSlider(
        value=DEFAULTS[p], base=10, min=-2, max=2, step=0.05,
        description=p, style=style, layout=widgets.Layout(width="350px"),
    )
    for p in PARAM_NAMES
}

sim_sliders = {
    "t_end": widgets.FloatSlider(value=30, min=5, max=100, step=1, description="t_end", style=style,
                                  layout=widgets.Layout(width="350px")),
    "t_on": widgets.FloatSlider(value=0, min=0, max=50, step=0.5, description="t_on", style=style,
                                 layout=widgets.Layout(width="350px")),
    "t_off": widgets.FloatSlider(value=5, min=0, max=50, step=0.5, description="t_off", style=style,
                                  layout=widgets.Layout(width="350px")),
}

# Group sliders by cascade layer
km_box = widgets.VBox([widgets.Label("Shared"), param_sliders["Km"]])
ras_box = widgets.VBox([widgets.Label("RAS"), param_sliders["k12"], param_sliders["k21"]])
raf_box = widgets.VBox([widgets.Label("RAF"), param_sliders["k34"], param_sliders["knfb"], param_sliders["k43"]])
mek_box = widgets.VBox([widgets.Label("MEK"), param_sliders["k56"], param_sliders["k65"]])
erk_box = widgets.VBox([widgets.Label("ERK"), param_sliders["k78"], param_sliders["k87"]])
nfb_box = widgets.VBox([widgets.Label("NFB"), param_sliders["f12"], param_sliders["f21"]])
sim_box = widgets.VBox([widgets.Label("Simulation"), *sim_sliders.values()])

left = widgets.VBox([km_box, ras_box, raf_box, mek_box])
right = widgets.VBox([erk_box, nfb_box, sim_box])
controls = widgets.HBox([left, right])

out = widgets.interactive_output(simulate_and_plot, {**param_sliders, **sim_sliders})
display(controls, out)

Output()

In [ ]:
import json

def save_params(b):
    current = {p: param_sliders[p].value for p in PARAM_NAMES}
    path = "egfr_params.json"
    with open(path, "w") as f:
        json.dump(current, f, indent=2)
    print(f"Saved to {path}: {current}")

save_btn = widgets.Button(description="Save params")
save_btn.on_click(save_params)
display(save_btn)

Button(description='Save params', style=ButtonStyle())